# LDA: Temporal Topic Analysis

Analyze topic evolution over time (2000–2025) using **existing best LDA models**
trained on all documents. Same approach as Top2Vec temporal analysis.

1. Get dominant topic per document from the single trained model
2. Group documents by year
3. Compute per-year topic prevalence, c-TF-IDF word evolution, coherence & IRBO

**Topics are consistent across all years** — no alignment needed.

In [1]:
import gc
import time
import pickle
import ast
import pandas as pd
import numpy as np
from pathlib import Path
from itertools import combinations
from sklearn.feature_extraction.text import CountVectorizer
from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

## Configuration

In [2]:
LIST_SUBJECT = ["cs", "math", "physics"]

BASE_DIR = Path("../../../../data/preprocess")
MODEL_DIR = Path("../../../../models/lda/tuning")
RESULT_DIR = Path("../../../../results/lda/temporal")
VERSION = "v1"

# IRBO config
TOP_N_WORDS = 10
RBO_P = 0.9

# Create output directories
for subject in LIST_SUBJECT:
    (RESULT_DIR / subject).mkdir(parents=True, exist_ok=True)

print(f"Subjects: {LIST_SUBJECT}")
print(f"Model directory: {MODEL_DIR}")
print(f"Results directory: {RESULT_DIR}")

Subjects: ['cs', 'math', 'physics']
Model directory: ../../../../models/lda/tuning
Results directory: ../../../../results/lda/temporal


## Helper Functions

In [3]:
def parse_bow_text(text_val):
    """Parse BOW text column: stored as Python list literals."""
    try:
        tokens = ast.literal_eval(text_val)
        if isinstance(tokens, list):
            return tokens
    except (ValueError, SyntaxError):
        pass
    return str(text_val).split()


def load_dataset(subject: str) -> pd.DataFrame:
    """Load dataset with year column. Converts list-literal text to space-separated."""
    file_path = BASE_DIR / subject / "bow" / f"{VERSION}.csv"
    df = pd.read_csv(file_path)
    df["submitted_date"] = pd.to_datetime(df["submitted_date"])
    df["year"] = df["submitted_date"].dt.year
    # Convert list-literal text to space-separated for CountVectorizer
    df["text"] = df["text"].apply(lambda x: " ".join(parse_bow_text(x)))
    return df


def get_dominant_topics(model, corpus):
    """
    Get the dominant (highest probability) topic for each document.
    Returns array of topic IDs.
    """
    dominant_topics = []
    for doc_bow in corpus:
        topic_dist = model.get_document_topics(doc_bow, minimum_probability=0.0)
        if topic_dist:
            dominant = max(topic_dist, key=lambda x: x[1])[0]
        else:
            dominant = 0
        dominant_topics.append(dominant)
    return np.array(dominant_topics)


def compute_ctfidf_per_year(
    df, topic_col="topic", text_col="text", top_n=10,
    global_tuning=True, evolution_tuning=True,
):
    """
    Compute c-TF-IDF per topic per year with optional tuning.
    
    Mirrors BERTopic's topics_over_time() tuning approach:
      - global_tuning: average each (year, topic) c-TF-IDF with the
        global (all-year) topic c-TF-IDF to anchor representations.
      - evolution_tuning: average each (year, topic) c-TF-IDF with
        the previous year (t-1) to smooth transitions.
    
    Returns:
        topic_words_per_year: dict of {(year, topic_id): [word1, word2, ...]}
    """
    years = sorted(df["year"].unique())
    topics = sorted(df[topic_col].unique())
    
    # Group documents by (year, topic) and concatenate
    groups = df.groupby(["year", topic_col])[text_col].apply(
        lambda x: " ".join(x)
    ).reset_index()
    groups.columns = ["year", "topic", "text"]
    
    # Build vocabulary across all groups
    vectorizer = CountVectorizer(stop_words="english")
    tf_matrix = vectorizer.fit_transform(groups["text"])
    vocab = vectorizer.get_feature_names_out()
    
    # Compute IDF: log(N / df_t) where N = number of groups, df_t = groups containing term
    n_groups = tf_matrix.shape[0]
    df_t = (tf_matrix > 0).sum(axis=0).A1  # document frequency per term
    idf = np.log((n_groups + 1) / (df_t + 1)) + 1  # smoothed IDF
    
    # TF-IDF per group
    tfidf_matrix = tf_matrix.multiply(idf).toarray()
    
    # --- Global c-TF-IDF (topic only, ignoring year) ---
    global_tfidf = None
    global_topic_to_idx = {}
    if global_tuning:
        global_groups = df.groupby(topic_col)[text_col].apply(
            lambda x: " ".join(x)
        ).reset_index()
        global_groups.columns = ["topic", "text"]
        global_groups = global_groups.sort_values("topic").reset_index(drop=True)
        global_tf = vectorizer.transform(global_groups["text"])
        global_tfidf = global_tf.multiply(idf).toarray()
        # L1 normalise global representation (matches BERTopic)
        global_row_sums = global_tfidf.sum(axis=1, keepdims=True)
        global_row_sums[global_row_sums == 0] = 1
        global_tfidf = global_tfidf / global_row_sums
        global_topic_to_idx = {
            int(t): i for i, t in enumerate(global_groups["topic"])
        }
    
    # --- L1 normalise per-year matrix (before tuning, matches BERTopic) ---
    if global_tuning or evolution_tuning:
        row_sums = tfidf_matrix.sum(axis=1, keepdims=True)
        row_sums[row_sums == 0] = 1
        tfidf_matrix = tfidf_matrix / row_sums
    
    # --- Build index: (year, topic) -> row index ---
    yt_to_idx = {}
    for idx in range(len(groups)):
        y = groups.iloc[idx]["year"]
        t = groups.iloc[idx]["topic"]
        yt_to_idx[(y, t)] = idx
    
    # --- Evolution tuning: average with t-1 ---
    if evolution_tuning:
        for yi in range(1, len(years)):
            curr_year = years[yi]
            prev_year = years[yi - 1]
            for topic in topics:
                curr_key = (curr_year, topic)
                prev_key = (prev_year, topic)
                if curr_key in yt_to_idx and prev_key in yt_to_idx:
                    ci = yt_to_idx[curr_key]
                    pi = yt_to_idx[prev_key]
                    tfidf_matrix[ci] = (tfidf_matrix[ci] + tfidf_matrix[pi]) / 2.0
    
    # --- Global tuning: average with global representation ---
    if global_tuning:
        for idx in range(len(groups)):
            topic = int(groups.iloc[idx]["topic"])
            if topic in global_topic_to_idx:
                gi = global_topic_to_idx[topic]
                tfidf_matrix[idx] = (tfidf_matrix[idx] + global_tfidf[gi]) / 2.0
    
    # --- Final L2 normalise before extracting top words ---
    row_norms = np.linalg.norm(tfidf_matrix, axis=1, keepdims=True)
    row_norms[row_norms == 0] = 1
    tfidf_matrix = tfidf_matrix / row_norms
    
    # Extract top words per (year, topic)
    topic_words_per_year = {}
    for idx in range(len(groups)):
        year = groups.iloc[idx]["year"]
        topic = groups.iloc[idx]["topic"]
        scores = tfidf_matrix[idx]
        top_indices = scores.argsort()[-top_n:][::-1]
        top_words = [vocab[i] for i in top_indices if scores[i] > 0]
        topic_words_per_year[(year, topic)] = top_words
    
    return topic_words_per_year


def calculate_coherence_for_words(topic_word_lists, texts_tokenized, dictionary):
    """Calculate C_v coherence given a list of topic word lists."""
    valid_topics = [tw for tw in topic_word_lists if len(tw) >= 2]
    if len(valid_topics) == 0:
        return 0.0
    cm = CoherenceModel(
        topics=valid_topics,
        texts=texts_tokenized,
        dictionary=dictionary,
        coherence='c_v',
        processes=7
    )
    return cm.get_coherence()



def rbo(list1, list2, p=0.9):
    if not list1 and not list2:
        return 1.0
    if not list1 or not list2:
        return 0.0

    # assign short (S) and long (L)
    if len(list1) <= len(list2):
        S, L = list1, list2
    else:
        S, L = list2, list1

    s, l = len(S), len(L)

    S_seen = set()
    L_seen = set()

    X = 0  # overlap
    rbo = 0.0
    disjoint = 0.0
    ext_term = 0.0

    for d in range(l):
        if d < s:
            s_item = S[d]
            S_seen.add(s_item)
        else:
            s_item = None

        l_item = L[d]
        L_seen.add(l_item)

        overlap_incr = 0

        if d < s:
            if s_item == l_item:
                overlap_incr = 1
            else:
                if s_item in L_seen:
                    overlap_incr += 1
                if l_item in S_seen:
                    overlap_incr += 1
        else:
            if l_item in S_seen:
                overlap_incr = 1

        X += overlap_incr

        if d < s:
            A_d = 2.0 * X / (len(S_seen) + len(L_seen))
        else:
            A_d = X / (d + 1)

        rbo += (1 - p) * (p ** d) * A_d

        if d < s:
            ext_term = A_d * (p ** (d + 1))
        else:
            X_s = X - overlap_incr if d == s else X_s
            disjoint += (1 - p) * (p ** d) * (
                X_s * (d + 1 - s) / ((d + 1) * s)
            )
            ext_term = (
                ((X - X_s) / (d + 1) + X_s / s)
                * (p ** (d + 1))
            )

        # optional optimization (safe)
        if p ** d < 1e-12:
            break

    return min(max(rbo + disjoint + ext_term, 0.0), 1.0)

def calculate_irbo(topics_words, p=0.9):
    """Calculate mean IRBO diversity."""
    if len(topics_words) < 2:
        return 0.0
    irbo_scores = []
    for (i, j) in combinations(range(len(topics_words)), 2):
        similarity = rbo(topics_words[i], topics_words[j], p=p)
        irbo_scores.append(1.0 - similarity)
    return np.mean(irbo_scores)


## Load Models & Data

Load each best LDA model, reconstruct the BoW corpus using the model's dictionary,
then get the dominant topic for every document.

In [4]:
all_models = {}
all_data = {}
all_years = {}

for subject in LIST_SUBJECT:
    print(f"\nLoading {subject}...")

    # Load model
    model_path = MODEL_DIR / subject / "best_model.pkl"
    with open(model_path, "rb") as f:
        model = pickle.load(f)
    all_models[subject] = model

    # Load data
    df = load_dataset(subject)

    # Reconstruct BoW corpus using model's id2word dictionary
    print(f"  Building BoW corpus with model's dictionary ({len(model.id2word)} words)...")
    start = time.time()
    texts_tokenized = [text.split() for text in df["text"].tolist()]  # already space-separated by load_dataset
    corpus = [model.id2word.doc2bow(tokens) for tokens in texts_tokenized]

    # Get dominant topic per document
    print(f"  Getting dominant topics for {len(df):,} documents...")
    dominant_topics = get_dominant_topics(model, corpus)
    df["topic"] = dominant_topics
    elapsed = time.time() - start

    all_data[subject] = df
    years = sorted(df["year"].unique())
    all_years[subject] = years

    n_topics = model.num_topics
    print(f"  {subject}: {len(df):,} docs, {n_topics} topics, "
          f"{len(years)} years ({years[0]}-{years[-1]}) [{elapsed:.1f}s]")

    # Topic size distribution
    topic_sizes = df["topic"].value_counts().sort_index()
    print(f"  Topic sizes: min={topic_sizes.min()}, "
          f"max={topic_sizes.max()}, mean={topic_sizes.mean():.0f}")

    del corpus, texts_tokenized
    gc.collect()

print(f"\n✅ All subjects loaded")


Loading cs...
  Building BoW corpus with model's dictionary (15355 words)...
  Getting dominant topics for 165,756 documents...
  cs: 165,756 docs, 50 topics, 26 years (2000-2025) [33.6s]
  Topic sizes: min=83, max=11663, mean=3315

Loading math...
  Building BoW corpus with model's dictionary (12173 words)...
  Getting dominant topics for 157,085 documents...
  math: 157,085 docs, 50 topics, 26 years (2000-2025) [28.8s]
  Topic sizes: min=134, max=8550, mean=3142

Loading physics...
  Building BoW corpus with model's dictionary (15919 words)...
  Getting dominant topics for 146,311 documents...
  physics: 146,311 docs, 50 topics, 26 years (2000-2025) [26.8s]
  Topic sizes: min=39, max=7534, mean=2926

✅ All subjects loaded


## Topic Prevalence Over Time

For each year, compute the proportion of documents belonging to each topic.

In [5]:
for subject in LIST_SUBJECT:
    df = all_data[subject]
    model = all_models[subject]
    years = all_years[subject]
    n_topics = model.num_topics

    prevalence_csv = RESULT_DIR / subject / "topic_prevalence.csv"

    print(f"\n{'='*70}")
    print(f"Topic Prevalence: {subject.upper()} ({n_topics} topics)")
    print(f"{'='*70}")

    # Get global topic words from LDA model
    global_topic_words = {}
    for tid in range(n_topics):
        words = [w for w, _ in model.show_topic(tid, topn=5)]
        global_topic_words[tid] = ", ".join(words)

    prevalence_rows = []

    for year in years:
        year_df = df[df["year"] == year]
        n_docs_year = len(year_df)
        topic_counts = year_df["topic"].value_counts()

        for topic_id in range(n_topics):
            count = topic_counts.get(topic_id, 0)
            proportion = count / n_docs_year if n_docs_year > 0 else 0.0

            prevalence_rows.append({
                "subject": subject,
                "year": year,
                "topic_id": topic_id,
                "doc_count": count,
                "total_docs_year": n_docs_year,
                "proportion": round(proportion, 6),
                "top_words": global_topic_words[topic_id],
            })

        active_topics = (topic_counts > 0).sum()
        top_topic = topic_counts.idxmax()
        top_count = topic_counts.max()
        print(f"  {year}: {n_docs_year:,} docs, {active_topics}/{n_topics} active, "
              f"top=T{top_topic} ({top_count} docs)")

    prevalence_df = pd.DataFrame(prevalence_rows)
    prevalence_df.to_csv(prevalence_csv, index=False)
    print(f"\n  Saved: {prevalence_csv} ({len(prevalence_df)} rows)")


Topic Prevalence: CS (50 topics)
  2000: 488 docs, 46/50 active, top=T15 (117 docs)
  2001: 594 docs, 41/50 active, top=T15 (86 docs)
  2002: 648 docs, 43/50 active, top=T15 (105 docs)
  2003: 825 docs, 50/50 active, top=T20 (108 docs)
  2004: 948 docs, 47/50 active, top=T15 (142 docs)
  2005: 1,000 docs, 50/50 active, top=T20 (125 docs)
  2006: 1,000 docs, 48/50 active, top=T15 (110 docs)
  2007: 1,000 docs, 48/50 active, top=T38 (104 docs)
  2008: 1,000 docs, 49/50 active, top=T20 (126 docs)
  2009: 1,000 docs, 48/50 active, top=T20 (120 docs)
  2010: 1,362 docs, 50/50 active, top=T20 (152 docs)
  2011: 1,622 docs, 48/50 active, top=T20 (171 docs)
  2012: 2,254 docs, 49/50 active, top=T38 (173 docs)
  2013: 2,719 docs, 50/50 active, top=T20 (225 docs)
  2014: 2,989 docs, 50/50 active, top=T38 (239 docs)
  2015: 3,345 docs, 50/50 active, top=T38 (282 docs)
  2016: 4,280 docs, 49/50 active, top=T38 (289 docs)
  2017: 5,534 docs, 50/50 active, top=T3 (391 docs)
  2018: 7,470 docs, 50/5

## Topic Word Evolution (c-TF-IDF per Year)

Compute c-TF-IDF for each (year, topic) to track how topic word
compositions change over time.

In [6]:
all_topic_words_per_year = {}

for subject in LIST_SUBJECT:
    df = all_data[subject]
    model = all_models[subject]
    n_topics = model.num_topics

    evolution_csv = RESULT_DIR / subject / "topic_word_evolution.csv"

    print(f"\n{'='*70}")
    print(f"Topic Word Evolution: {subject.upper()}")
    print(f"{'='*70}")

    start = time.time()
    topic_words_per_year = compute_ctfidf_per_year(
        df, topic_col="topic", text_col="text", top_n=TOP_N_WORDS,
        global_tuning=False, evolution_tuning=True,
    )
    elapsed = time.time() - start
    all_topic_words_per_year[subject] = topic_words_per_year

    print(f"  c-TF-IDF computed in {elapsed:.1f}s")
    print(f"  (year, topic) groups: {len(topic_words_per_year)}")

    evolution_rows = []
    for (year, topic_id), words in sorted(topic_words_per_year.items()):
        evolution_rows.append({
            "subject": subject,
            "year": year,
            "topic_id": topic_id,
            "top_words": ", ".join(words),
        })

    evolution_df = pd.DataFrame(evolution_rows)
    evolution_df.to_csv(evolution_csv, index=False)
    print(f"  Saved: {evolution_csv}")

    # Example: topic 0 across years
    print(f"\n  Example — Topic 0 word evolution:")
    for year in [2000, 2005, 2010, 2015, 2020, 2025]:
        key = (year, 0)
        if key in topic_words_per_year:
            words = ", ".join(topic_words_per_year[key][:5])
            print(f"    {year}: {words}")


Topic Word Evolution: CS
  c-TF-IDF computed in 10.4s
  (year, topic) groups: 1266
  Saved: ../../../../results/lda/temporal/cs/topic_word_evolution.csv

  Example — Topic 0 word evolution:
    2000: belief_revision, reasoning, snep, nonmonotonic, knowledge
    2005: treillis, knowledge, concept, sources, ontology
    2010: knowledge, ontology, kra, theory, language
    2015: ontology, knowledge, question, concept, reasoning
    2020: question, knowledge, reasoning, answer, dataset
    2025: llm, reasoning, knowledge, language, question

Topic Word Evolution: MATH
  c-TF-IDF computed in 5.9s
  (year, topic) groups: 1289
  Saved: ../../../../results/lda/temporal/math/topic_word_evolution.csv

  Example — Topic 0 word evolution:
    2000: theory, logic, sedenion, theorem, proof
    2005: theory, theorem, proof, geometry, group
    2010: theory, proof, theorem, group, logic
    2015: theory, logic, proof, theorem, mathematical
    2020: theory, logic, theorem, proof, mathematical
    202

## Per-Year Coherence & IRBO

For each year, compute coherence and IRBO using the year-specific
c-TF-IDF topic words.

In [7]:
for subject in LIST_SUBJECT:
    df = all_data[subject]
    model = all_models[subject]
    years = all_years[subject]
    n_topics = model.num_topics
    topic_words_per_year = all_topic_words_per_year[subject]

    metrics_csv = RESULT_DIR / subject / "per_year_metrics.csv"

    print(f"\n{'='*70}")
    print(f"Per-Year Metrics: {subject.upper()} ({n_topics} topics)")
    print(f"{'='*70}")

    # Build corpus-level tokenized texts and dictionary (full corpus as reference)
    corpus_texts_tokenized = [text.split() for text in df["text"].tolist()]
    corpus_dictionary = Dictionary(corpus_texts_tokenized)
    print(f"  Corpus: {len(corpus_texts_tokenized):,} docs, {len(corpus_dictionary):,} vocab")

    metrics_rows = []

    for year in years:
        year_df = df[df["year"] == year]
        n_docs = len(year_df)

        active_topics = sorted(year_df["topic"].unique())
        year_topic_words = []
        for tid in active_topics:
            key = (year, tid)
            if key in topic_words_per_year and len(topic_words_per_year[key]) >= 2:
                year_topic_words.append(topic_words_per_year[key])

        coherence = calculate_coherence_for_words(
            year_topic_words, corpus_texts_tokenized, corpus_dictionary
        )
        irbo_mean = calculate_irbo(year_topic_words, p=RBO_P)

        if coherence + irbo_mean > 0:
            topic_quality = 2 * coherence * irbo_mean / (coherence + irbo_mean)
        else:
            topic_quality = 0.0

        n_active = len(active_topics)
        print(f"  {year}: {n_docs:,} docs, {n_active} active | "
              f"Quality={topic_quality:.4f} (C={coherence:.4f}, IRBO={irbo_mean:.4f})")

        metrics_rows.append({
            "subject": subject,
            "year": year,
            "num_docs": n_docs,
            "num_topics_total": n_topics,
            "num_topics_active": n_active,
            "coherence_cv": round(coherence, 6),
            "irbo_mean": round(irbo_mean, 6),
            "topic_quality": round(topic_quality, 6),
        })

    metrics_df = pd.DataFrame(metrics_rows)
    metrics_df.to_csv(metrics_csv, index=False)
    print(f"\n  Saved: {metrics_csv}")


Per-Year Metrics: CS (50 topics)
  Corpus: 165,756 docs, 146,603 vocab
  2000: 488 docs, 46 active | Quality=0.5538 (C=0.3838, IRBO=0.9940)
  2001: 594 docs, 41 active | Quality=0.5557 (C=0.3857, IRBO=0.9936)
  2002: 648 docs, 43 active | Quality=0.6004 (C=0.4306, IRBO=0.9913)
  2003: 825 docs, 50 active | Quality=0.5837 (C=0.4138, IRBO=0.9908)
  2004: 948 docs, 47 active | Quality=0.5924 (C=0.4227, IRBO=0.9894)
  2005: 1,000 docs, 50 active | Quality=0.6243 (C=0.4561, IRBO=0.9894)
  2006: 1,000 docs, 48 active | Quality=0.6430 (C=0.4765, IRBO=0.9883)
  2007: 1,000 docs, 48 active | Quality=0.6208 (C=0.4518, IRBO=0.9917)
  2008: 1,000 docs, 49 active | Quality=0.6251 (C=0.4566, IRBO=0.9905)
  2009: 1,000 docs, 48 active | Quality=0.6009 (C=0.4313, IRBO=0.9903)
  2010: 1,362 docs, 50 active | Quality=0.6259 (C=0.4581, IRBO=0.9881)
  2011: 1,622 docs, 48 active | Quality=0.6172 (C=0.4494, IRBO=0.9852)
  2012: 2,254 docs, 49 active | Quality=0.6439 (C=0.4792, IRBO=0.9809)
  2013: 2,719 d

## Topic Trends: Growing, Stable, and Declining

In [8]:
from scipy.stats import linregress

for subject in LIST_SUBJECT:
    df = all_data[subject]
    model = all_models[subject]
    # Get topic words from evolution CSV (use latest available year per topic)
    evo_path = RESULT_DIR / subject / "topic_word_evolution.csv"
    evo_df = pd.read_csv(evo_path)
    global_tw = {}
    for tid in range(model.num_topics):
        tid_evo = evo_df[evo_df['topic_id'] == tid].sort_values('year', ascending=False)
        if len(tid_evo) > 0:
            global_tw[tid] = [w.strip() for w in str(tid_evo.iloc[0]['top_words']).split(',')][:5]
        else:
            # Fallback to LDA model's own word distribution
            global_tw[tid] = [w for w, _ in model.show_topic(tid, topn=5)]

    rows = []
    for tid in range(model.num_topics):
        topic_df = df[df["topic"] == tid]
        if len(topic_df) == 0:
            continue

        year_counts = topic_df["year"].value_counts().sort_index()
        total_per_year = df["year"].value_counts().sort_index()
        proportions = (year_counts / total_per_year).fillna(0)

        # Align proportions with all years
        all_years = sorted(total_per_year.index)
        prop_aligned = proportions.reindex(all_years, fill_value=0.0)

        years_arr = np.array(all_years, dtype=float)
        props_arr = prop_aligned.values.astype(float)

        # Linear regression
        slope, intercept, r_val, p_val, std_err = linregress(years_arr, props_arr)

        # Early/late for display
        topic_years = sorted(year_counts.index)
        early_mean = proportions[topic_years[:5]].mean() if len(topic_years) >= 5 else proportions.mean()
        late_mean = proportions[topic_years[-5:]].mean() if len(topic_years) >= 5 else proportions.mean()

        # Classify by slope significance
        if p_val < 0.05 and slope > 0:
            trend_label = "GROWING"
        elif p_val < 0.05 and slope < 0:
            trend_label = "DECLINING"
        else:
            trend_label = "STABLE"

        top_words = global_tw.get(tid, [w for w, _ in model.show_topic(tid, topn=5)])
        rows.append({
            "subject": subject, "topic_id": tid,
            "top_words": ", ".join(top_words),
            "total_docs": len(topic_df),
            "first_year": year_counts.index.min(),
            "last_year": year_counts.index.max(),
            "early_proportion": round(early_mean, 6),
            "late_proportion": round(late_mean, 6),
            "slope": round(slope, 8),
            "r_squared": round(r_val**2, 4),
            "p_value": round(p_val, 6),
            "trend": trend_label,
        })

    trends_df = pd.DataFrame(rows)
    trends_df.to_csv(RESULT_DIR / subject / "topic_trends.csv", index=False)

    g = len(trends_df[trends_df["trend"] == "GROWING"])
    s = len(trends_df[trends_df["trend"] == "STABLE"])
    d = len(trends_df[trends_df["trend"] == "DECLINING"])
    print(f"  {subject.upper()}: Growing={g}, Stable={s}, Declining={d}")


  CS: Growing=19, Stable=19, Declining=12
  MATH: Growing=16, Stable=11, Declining=23
  PHYSICS: Growing=18, Stable=16, Declining=16


## Top 5 Growing & Declining Topics

In [9]:
for subject in LIST_SUBJECT:
    trends_df = pd.read_csv(RESULT_DIR / subject / "topic_trends.csv")

    print(f"\n{'='*80}")
    print(f"  {subject.upper()}")
    print(f"{'='*80}")

    growing = trends_df[trends_df['trend'] == 'GROWING'].sort_values('slope', ascending=False)
    declining = trends_df[trends_df['trend'] == 'DECLINING'].sort_values('slope', ascending=True)

    print(f"\n  " + chr(0x1F4C8) + f" TOP 5 GROWING (steepest positive slope):")
    for _, row in growing.head(5).iterrows():
        print(f"    T{int(row['topic_id']):>3} | slope={row['slope']:+.6f} "
              f"R\u00B2={row['r_squared']:.3f} | "
              f"{row['early_proportion']:.4f} \u2192 {row['late_proportion']:.4f} | "
              f"{row['top_words']}")

    print(f"\n  " + chr(0x1F4C9) + f" TOP 5 DECLINING (steepest negative slope):")
    for _, row in declining.head(5).iterrows():
        print(f"    T{int(row['topic_id']):>3} | slope={row['slope']:+.6f} "
              f"R\u00B2={row['r_squared']:.3f} | "
              f"{row['early_proportion']:.4f} \u2192 {row['late_proportion']:.4f} | "
              f"{row['top_words']}")



  CS

  📈 TOP 5 GROWING (steepest positive slope):
    T 17 | slope=+0.003511 R²=0.750 | 0.0152 → 0.0872 | datum, dataset, training, label, learning
    T  3 | slope=+0.003419 R²=0.794 | 0.0073 → 0.0672 | image, segmentation, dataset, scene, reconstruction
    T 35 | slope=+0.002891 R²=0.665 | 0.0036 → 0.0680 | image, domain, visual, diffusion, representation
    T 12 | slope=+0.001425 R²=0.589 | 0.0042 → 0.0250 | neural_network, layer, network, training, deep
    T 34 | slope=+0.001327 R²=0.758 | 0.0083 → 0.0338 | robot, motion, trajectory, environment, navigation

  📉 TOP 5 DECLINING (steepest negative slope):
    T 15 | slope=-0.006637 R²=0.840 | 0.1640 → 0.0166 | logic, tree, language, proof, semantic
    T 20 | slope=-0.004981 R²=0.830 | 0.1160 → 0.0208 | polynomial, graph, function, class, complexity
    T 39 | slope=-0.001920 R²=0.632 | 0.0468 → 0.0148 | distribution, uncertainty, probability, bayesian, bound
    T 37 | slope=-0.001071 R²=0.691 | 0.0361 → 0.0118 | quantum, prog

## Evolution Summary

In [10]:
for subject in LIST_SUBJECT:
    metrics_csv = RESULT_DIR / subject / "per_year_metrics.csv"
    trends_csv = RESULT_DIR / subject / "topic_trends.csv"

    if not metrics_csv.exists():
        continue

    metrics_df = pd.read_csv(metrics_csv)
    trends_df = pd.read_csv(trends_csv)
    n_topics = all_models[subject].num_topics

    growing = len(trends_df[trends_df["trend"] == "GROWING"])
    declining = len(trends_df[trends_df["trend"] == "DECLINING"])
    stable = len(trends_df[trends_df["trend"] == "STABLE"])

    summary_data = {
        "subject": subject,
        "num_topics": n_topics,
        "num_years": len(metrics_df),
        "coherence_mean": round(metrics_df["coherence_cv"].mean(), 6),
        "coherence_std": round(metrics_df["coherence_cv"].std(), 6),
        "irbo_mean": round(metrics_df["irbo_mean"].mean(), 6),
        "irbo_std": round(metrics_df["irbo_mean"].std(), 6),
        "quality_mean": round(metrics_df["topic_quality"].mean(), 6),
        "quality_std": round(metrics_df["topic_quality"].std(), 6),
        "topics_growing": growing,
        "topics_stable": stable,
        "topics_declining": declining,
    }
    summary_df = pd.DataFrame([summary_data])
    summary_csv = RESULT_DIR / subject / "evolution_summary.csv"
    summary_df.to_csv(summary_csv, index=False)
    print(f"  {subject.upper()} summary saved: {summary_csv}")

  CS summary saved: ../../../../results/lda/temporal/cs/evolution_summary.csv
  MATH summary saved: ../../../../results/lda/temporal/math/evolution_summary.csv
  PHYSICS summary saved: ../../../../results/lda/temporal/physics/evolution_summary.csv


## Final Results

In [11]:
print("\n" + "=" * 110)
print("LDA TEMPORAL ANALYSIS: FINAL RESULTS")
print("=" * 110)

for subject in LIST_SUBJECT:
    metrics_csv = RESULT_DIR / subject / "per_year_metrics.csv"
    trends_csv = RESULT_DIR / subject / "topic_trends.csv"

    if not metrics_csv.exists():
        print(f"\n{subject.upper()}: No results found")
        continue

    metrics_df = pd.read_csv(metrics_csv)
    trends_df = pd.read_csv(trends_csv)
    n_topics = all_models[subject].num_topics

    growing = len(trends_df[trends_df["trend"] == "GROWING"])
    declining = len(trends_df[trends_df["trend"] == "DECLINING"])
    stable = len(trends_df[trends_df["trend"] == "STABLE"])

    sep = chr(9472)
    print(f"\n{sep*60}")
    print(f"  Subject:        {subject.upper()}")
    print(f"  Num topics:     {n_topics}")
    print(f"  Years:          {len(metrics_df)}")
    print(f"  Coherence:      {metrics_df['coherence_cv'].mean():.4f} +/- {metrics_df['coherence_cv'].std():.4f}")
    print(f"  IRBO:           {metrics_df['irbo_mean'].mean():.4f} +/- {metrics_df['irbo_mean'].std():.4f}")
    print(f"  Topic Quality:  {metrics_df['topic_quality'].mean():.4f} +/- {metrics_df['topic_quality'].std():.4f}")
    print(f"  Trends:         ↑{growing} growing, →{stable} stable, ↓{declining} declining")
    print(f"{sep*60}")

print("\n" + "=" * 110)
print("Per-Year Details:")
print("=" * 110)

for subject in LIST_SUBJECT:
    metrics_csv = RESULT_DIR / subject / "per_year_metrics.csv"
    if not metrics_csv.exists():
        continue
    metrics_df = pd.read_csv(metrics_csv)
    print(f"\n{subject.upper()}:")
    print(metrics_df[["year", "num_docs", "num_topics_active",
                     "coherence_cv", "irbo_mean", "topic_quality"]].to_string(index=False))
    print()


LDA TEMPORAL ANALYSIS: FINAL RESULTS

────────────────────────────────────────────────────────────
  Subject:        CS
  Num topics:     50
  Years:          26
  Coherence:      0.5196 +/- 0.0897
  IRBO:           0.9824 +/- 0.0081
  Topic Quality:  0.6749 +/- 0.0752
  Trends:         ↑19 growing, →19 stable, ↓12 declining
────────────────────────────────────────────────────────────

────────────────────────────────────────────────────────────
  Subject:        MATH
  Num topics:     50
  Years:          26
  Coherence:      0.5640 +/- 0.0648
  IRBO:           0.9890 +/- 0.0024
  Topic Quality:  0.7160 +/- 0.0547
  Trends:         ↑16 growing, →11 stable, ↓23 declining
────────────────────────────────────────────────────────────

────────────────────────────────────────────────────────────
  Subject:        PHYSICS
  Num topics:     50
  Years:          26
  Coherence:      0.5913 +/- 0.0740
  IRBO:           0.9858 +/- 0.0047
  Topic Quality:  0.7363 +/- 0.0597
  Trends:         ↑1